In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH  = "/content/drive/MyDrive/MSc_Dissertation/data/"
MODEL_PATH = "/content/drive/MyDrive/MSc_Dissertation/models/"
FIG_PATH   = "/content/drive/MyDrive/MSc_Dissertation/figures/"

import os
os.makedirs(FIG_PATH, exist_ok=True)
!pip install xgboost imbalanced-learn shap lime -q

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap, lime, lime.lime_tabular, pickle, warnings
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
warnings.filterwarnings('ignore')
np.random.seed(42)

DPI = 300
def save(name):
    plt.savefig(FIG_PATH + name, dpi=DPI, bbox_inches='tight'); plt.close()
    print(f"  saved: {name}")

print("Loading artefacts...")
xgb_model = pickle.load(open(MODEL_PATH+'xgboost_tuned.pkl','rb'))
rf_model  = pickle.load(open(MODEL_PATH+'random_forest_tuned.pkl','rb'))
lr_model  = pickle.load(open(MODEL_PATH+'logistic_regression_tuned.pkl','rb'))
test_data = pickle.load(open(MODEL_PATH+'test_data.pkl','rb'))
shap_out  = pickle.load(open(MODEL_PATH+'shap_outputs.pkl','rb'))
seg       = pickle.load(open(MODEL_PATH+'segmentation.pkl','rb'))

X_test_scaled = test_data['X_test_scaled']; y_test = test_data['y_test']
feature_names = test_data['feature_names']; n_feat = len(feature_names)

xgb_shap = shap_out['xgb_shap']; xgb_base = shap_out['xgb_base']
rf_shap  = shap_out['rf_shap'];  lr_shap  = shap_out['lr_shap']
examples = shap_out['examples']
colors = {'Logistic Regression':'#3498db','Random Forest':'#2ecc71','XGBoost':'#e74c3c'}
models = {'Logistic Regression':lr_model,'Random Forest':rf_model,'XGBoost':xgb_model}
preds  = {n:m.predict_proba(X_test_scaled)[:,1] for n,m in models.items()}
probs  = pd.Series(preds['XGBoost'], index=X_test_scaled.index)
print("Loaded. Feature count:", n_feat)

Loading artefacts...
Loaded. Feature count: 66


In [ ]:
print("DAY 1 figures:")

# 01 ROC curves
fig = plt.figure(figsize=(10,7))
for n in models:
    fpr,tpr,_ = roc_curve(y_test, preds[n])
    plt.plot(fpr,tpr,color=colors[n],lw=2.5,label=f"{n} (AUC={roc_auc_score(y_test,preds[n]):.3f})")
plt.plot([0,1],[0,1],'k--',alpha=.3); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Tuned Models',fontweight='bold'); plt.legend(loc='lower right'); plt.grid(alpha=.3)
save('01_roc_curves.png')

# 02 confusion matrices
fig, axes = plt.subplots(1,3,figsize=(20,5))
for ax,n in zip(axes,models):
    cm = confusion_matrix(y_test,(preds[n]>=0.5).astype(int))
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=ax,
                xticklabels=['Non-Churn','Churn'],yticklabels=['Non-Churn','Churn'])
    ax.set_title(n,fontweight='bold'); ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.suptitle('Confusion Matrices',fontweight='bold',y=1.02); save('02_confusion_matrices.png')

# 03 metric comparison bar
res = pd.read_csv(MODEL_PATH+'model_comparison_results.csv',index_col=0)
fig = plt.figure(figsize=(12,6)); xp = np.arange(len(res.columns)); w=0.25
for i,(mdl,col) in enumerate(zip(res.index,colors.values())):
    plt.bar(xp+i*w,res.loc[mdl].values,w,label=mdl,color=col,edgecolor='black',alpha=.85)
plt.xticks(xp+w,res.columns); plt.ylim(0,1.05)
plt.title('Model Performance Comparison',fontweight='bold'); plt.legend(); plt.grid(axis='y',alpha=.3)
save('03_model_comparison_bar.png')

# 04 built-in importances
fig, axes = plt.subplots(1,2,figsize=(20,8))
pd.Series(xgb_model.feature_importances_,index=feature_names).sort_values().tail(20).plot(
    kind='barh',ax=axes[0],color='#e74c3c',edgecolor='black'); axes[0].set_title('Top 20 — XGBoost',fontweight='bold')
pd.Series(rf_model.feature_importances_,index=feature_names).sort_values().tail(20).plot(
    kind='barh',ax=axes[1],color='#2ecc71',edgecolor='black'); axes[1].set_title('Top 20 — Random Forest',fontweight='bold')
save('04_feature_importance_xgb_rf.png')

DAY 1 figures:
  saved: 01_roc_curves.png
  saved: 02_confusion_matrices.png
  saved: 03_model_comparison_bar.png
  saved: 04_feature_importance_xgb_rf.png


In [ ]:
print("DAY 2 SHAP global figures:")

# 05 beeswarm
plt.figure(figsize=(12,10))
shap.summary_plot(xgb_shap, X_test_scaled, feature_names=feature_names, max_display=20, show=False)
plt.title("SHAP Beeswarm — XGBoost (Top 20)",fontsize=14,fontweight='bold'); save('05_shap_beeswarm_xgb.png')

# 06 bar
plt.figure(figsize=(12,8))
shap.summary_plot(xgb_shap, X_test_scaled, feature_names=feature_names, plot_type="bar", max_display=20, show=False)
plt.title("Mean |SHAP| — XGBoost (Top 20)",fontsize=14,fontweight='bold'); save('06_shap_bar_xgb.png')

# 07 normalised importance across 3 models  (uses saved importance_norm)
imp = shap_out['importance_norm'].copy()
imp.columns = ['XGBoost','RandomForest','LogReg'][:imp.shape[1]]
xgb_top = imp['XGBoost'].sort_values(ascending=False)
plt.figure(figsize=(12,9))
imp.loc[xgb_top.head(15).index].plot(kind='barh',ax=plt.gca(),
    color=['#e74c3c','#2ecc71','#3498db'],edgecolor='black')
plt.gca().invert_yaxis(); plt.title('Normalised Importance — XGB vs RF vs LR (top 15)',fontweight='bold')
plt.xlabel('Share of total |SHAP|'); save('07_shap_importance_xgb_rf_lr.png')

# 12 dependence top 5
top5 = xgb_top.index[:5].tolist()
fig, axes = plt.subplots(1,5,figsize=(25,5))
for ax,f in zip(axes,top5):
    shap.dependence_plot(f,xgb_shap,X_test_scaled,feature_names=feature_names,ax=ax,show=False)
    ax.set_title(f,fontweight='bold')
plt.suptitle('SHAP Dependence — Top 5 (XGBoost)',fontsize=15,fontweight='bold'); save('12_shap_dependence_top5.png')

# 13 interaction top 2
t2 = top5[:2]
fig, axes = plt.subplots(1,2,figsize=(16,6))
shap.dependence_plot(t2[0],xgb_shap,X_test_scaled,interaction_index=t2[1],feature_names=feature_names,ax=axes[0],show=False)
axes[0].set_title(f'{t2[0]} x {t2[1]}',fontweight='bold')
shap.dependence_plot(t2[1],xgb_shap,X_test_scaled,interaction_index=t2[0],feature_names=feature_names,ax=axes[1],show=False)
axes[1].set_title(f'{t2[1]} x {t2[0]}',fontweight='bold')
plt.suptitle('Feature Interactions — Top 2 Drivers',fontsize=14,fontweight='bold'); save('13_shap_interaction_top2.png')

DAY 2 SHAP global figures:
  saved: 05_shap_beeswarm_xgb.png
  saved: 06_shap_bar_xgb.png
  saved: 07_shap_importance_xgb_rf_lr.png
  saved: 12_shap_dependence_top5.png
  saved: 13_shap_interaction_top2.png


In [ ]:
print("DAY 2 SHAP local waterfalls (probability space)...")
background = shap.sample(X_test_scaled, 100, random_state=42)
xgb_prob_expl = shap.KernelExplainer(lambda x: xgb_model.predict_proba(x)[:,1], background, link='identity')
prob_base = xgb_prob_expl.expected_value

fmap = {'High-Risk Churner':'08_shap_waterfall_high_risk.png',
        'Borderline Churner':'09_shap_waterfall_borderline.png',
        'Confident Non-Churner':'10_shap_waterfall_confident_nonchurner.png',
        'Missed Churner (False Negative)':'11_shap_waterfall_missed_churner.png'}
for label, idx in examples.items():
    row = X_test_scaled.loc[[idx]]
    sv = xgb_prob_expl.shap_values(row)[0]
    plt.figure(figsize=(12,6))
    shap.waterfall_plot(shap.Explanation(values=sv, base_values=prob_base,
                        data=row.values[0], feature_names=feature_names), max_display=12, show=False)
    plt.title(f'SHAP (prob) — {label}\nP(Churn)={probs[idx]:.3f}',fontsize=12,fontweight='bold')
    save(fmap[label])

DAY 2 SHAP local waterfalls (probability space)...


  0%|          | 0/1 [00:00<?, ?it/s]

  saved: 08_shap_waterfall_high_risk.png


  0%|          | 0/1 [00:00<?, ?it/s]

  saved: 09_shap_waterfall_borderline.png


  0%|          | 0/1 [00:00<?, ?it/s]

  saved: 10_shap_waterfall_confident_nonchurner.png


  0%|          | 0/1 [00:00<?, ?it/s]

  saved: 11_shap_waterfall_missed_churner.png


In [ ]:
print("DAY 3 LIME figures...")
cat_idx = [i for i,f in enumerate(feature_names) if X_test_scaled[f].nunique()<=2]
lime_expl = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_test_scaled.values, feature_names=feature_names,
    class_names=['Non-Churn','Churn'], categorical_features=cat_idx,
    mode='classification', discretize_continuous=False, random_state=42)
lime_pred = lambda x: xgb_model.predict_proba(pd.DataFrame(x,columns=feature_names))

fmap = {'High-Risk Churner':'14_lime_local_high_risk.png',
        'Borderline Churner':'15_lime_local_borderline.png',
        'Confident Non-Churner':'16_lime_local_confident_nonchurner.png',
        'Missed Churner (False Negative)':'17_lime_local_missed_churner.png'}
for label, idx in examples.items():
    e = lime_expl.explain_instance(X_test_scaled.loc[idx].values, lime_pred,
            num_features=12, labels=(1,), num_samples=3000)
    fig = e.as_pyplot_figure(label=1); fig.set_size_inches(10,6)
    plt.title(f'LIME — {label}\nP(Churn)={probs[idx]:.3f}',fontsize=12,fontweight='bold')
    save(fmap[label])

# 18 SHAP vs LIME importance
lime_eval = pd.read_csv(MODEL_PATH+'lime_eval.csv',index_col=0).reindex(columns=feature_names)
shap_eval = pd.read_csv(MODEL_PATH+'xgb_shap_eval.csv',index_col=0).reindex(columns=feature_names)
comp = pd.DataFrame({'SHAP':shap_eval.abs().mean(),'LIME':lime_eval.abs().mean()})
for c in comp.columns: comp[c] = comp[c]/comp[c].sum()
plt.figure(figsize=(11,9))
comp.loc[comp['SHAP'].sort_values(ascending=False).head(15).index].plot(
    kind='barh',ax=plt.gca(),color=['#9b59b6','#f39c12'],edgecolor='black')
plt.gca().invert_yaxis(); plt.title('XGBoost: SHAP vs LIME importance (top 15 by SHAP)',fontweight='bold')
plt.xlabel('Share of total |attribution|'); save('18_shap_vs_lime_importance.png')

DAY 3 LIME figures...
  saved: 14_lime_local_high_risk.png
  saved: 15_lime_local_borderline.png
  saved: 16_lime_local_confident_nonchurner.png
  saved: 17_lime_local_missed_churner.png
  saved: 18_shap_vs_lime_importance.png


In [ ]:
print("DAY 4 evaluation figures...")
shap_eval = pd.read_csv(MODEL_PATH+'xgb_shap_eval.csv',index_col=0).reindex(columns=feature_names)
lime_eval = pd.read_csv(MODEL_PATH+'lime_eval.csv',index_col=0).reindex(columns=feature_names)
lime_eval.index = lime_eval.index.astype(shap_eval.index.dtype)
eval_idx = shap_eval.index
baseline = X_test_scaled[feature_names].mean().values
Xe = X_test_scaled.loc[eval_idx, feature_names].values
p0 = xgb_model.predict_proba(X_test_scaled.loc[eval_idx])[:,1]
FID_K = 15
def dcurve(attr):
    order = np.argsort(-np.abs(attr),axis=1); Xm = Xe.copy(); curve=[p0.copy()]; rows=np.arange(len(Xm))
    for s in range(FID_K):
        f=order[:,s]; Xm[rows,f]=baseline[f]
        curve.append(xgb_model.predict_proba(pd.DataFrame(Xm,columns=feature_names))[:,1])
    curve=np.array(curve); return np.abs(curve[0:1]-curve).mean(), curve.mean(axis=1)
rng=np.random.default_rng(42)
sa,sc = dcurve(shap_eval.values); la,lc = dcurve(lime_eval.values); ra,rc = dcurve(rng.standard_normal(shap_eval.shape))

# 19 fidelity deletion curves
plt.figure(figsize=(10,6)); xs=np.arange(FID_K+1)
plt.plot(xs,sc,'o-',color='#9b59b6',label=f'SHAP (AOPC {sa:.3f})')
plt.plot(xs,lc,'s-',color='#f39c12',label=f'LIME (AOPC {la:.3f})')
plt.plot(xs,rc,'^--',color='gray',label=f'Random (AOPC {ra:.3f})')
plt.xlabel('# top features removed'); plt.ylabel('Mean P(churn)')
plt.title('Fidelity — deletion curves',fontweight='bold'); plt.legend(); plt.grid(alpha=.3)
save('19_fidelity_deletion_curves.png')

# 20 sparsity histogram
def sparsity(df):
    A=np.abs(df.values); thr=0.05*A.max(axis=1,keepdims=True); return (A>thr).sum(axis=1)
ss, ls = sparsity(shap_eval), sparsity(lime_eval)
plt.figure(figsize=(10,5)); bins=np.arange(0,max(ss.max(),ls.max())+2)
plt.hist(ss,bins=bins,alpha=.6,label=f'SHAP (mean {ss.mean():.1f})',color='#9b59b6')
plt.hist(ls,bins=bins,alpha=.6,label=f'LIME (mean {ls.mean():.1f})',color='#f39c12')
plt.axvspan(5,7,color='green',alpha=.12,label='marketer-ideal 5-7')
plt.xlabel('Active features per explanation'); plt.ylabel('Customers')
plt.title('Sparsity — SHAP vs LIME',fontweight='bold'); plt.legend(); save('20_sparsity_histogram.png')

DAY 4 evaluation figures...
  saved: 19_fidelity_deletion_curves.png
  saved: 20_sparsity_histogram.png


In [ ]:
print("DAY 5 segmentation figures...")
actual_idx = pd.Index(seg['actual_idx']); labels = np.asarray(seg['actual_labels'])
bk = seg['best_k']; segdf = seg['segment_df']; cp = seg['cluster_profiles']

# rebuild the SHAP-vector space for actual churners (for silhouette sweep + scatter)
S = pd.read_csv(MODEL_PATH+'xgb_shap_values.csv',index_col=0).reindex(columns=feature_names)
S.index = S.index.astype(actual_idx.dtype)
Sz = seg['scaler'].transform(S.loc[actual_idx].values)
Spca = seg['pca'].transform(Sz)   # same 10-D PCA used for clustering

# 21 silhouette sweep (rebuilt) + elbow
sil, inertia = {}, {}
for k in range(2,9):
    km = KMeans(k, random_state=42, n_init=10).fit(Spca)
    sil[k] = silhouette_score(Spca, km.labels_); inertia[k] = km.inertia_
fig, ax = plt.subplots(1,2,figsize=(15,5))
ax[0].plot(list(sil),list(sil.values()),'o-',color='#9b59b6')
ax[0].axhline(0.15,ls='--',color='red',alpha=.6,label='gate 0.15')
ax[0].axvline(bk,ls=':',color='green',label=f'chosen k={bk}')
ax[0].set_xlabel('k'); ax[0].set_ylabel('Silhouette'); ax[0].set_title('Silhouette by k (PCA-space)',fontweight='bold'); ax[0].legend()
ax[1].plot(list(inertia),list(inertia.values()),'o-',color='#3498db')
ax[1].set_xlabel('k'); ax[1].set_ylabel('Inertia'); ax[1].set_title('Elbow',fontweight='bold')
save('21_silhouette_elbow.png')

# 22 cluster reason-profile heatmap
top_union = pd.Index(pd.concat([cp.loc[c].abs().sort_values(ascending=False).head(6) for c in range(bk)]).index.unique())
plt.figure(figsize=(min(1.1*len(top_union),18),0.8*bk+2))
sns.heatmap(cp[top_union],cmap='RdBu_r',center=0,annot=True,fmt='.2f',cbar_kws={'label':'mean SHAP (->churn if +)'})
plt.title('Cluster Reason-Profiles — mean SHAP per driver',fontweight='bold'); plt.ylabel('Cluster'); plt.xlabel('Feature')
save('22_cluster_profiles_heatmap.png')

# 23 PCA segment scatter (2-D view of the cluster space)
coords = PCA(n_components=2, random_state=42).fit_transform(Sz)
plt.figure(figsize=(9,7))
for c in range(bk):
    m = labels==c
    plt.scatter(coords[m,0],coords[m,1],s=12,alpha=.5,label=f"C{c}: {segdf.loc[c,'segment']}")
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('Churn Segments in SHAP-PCA Space',fontweight='bold')
plt.legend(fontsize=8); save('23_segments_pca_scatter.png')

DAY 5 segmentation figures...
  saved: 21_silhouette_elbow.png
  saved: 22_cluster_profiles_heatmap.png
  saved: 23_segments_pca_scatter.png


In [ ]:
saved = sorted(os.listdir(FIG_PATH))
print("="*50); print(f"FIGURES IN {FIG_PATH}: {len(saved)}"); print("="*50)
for f in saved: print("  ", f)
print("\nExpected 23. Download the figures/ folder -> code\\outputs\\figures\\")

FIGURES IN /content/drive/MyDrive/MSc_Dissertation/figures/: 23
   01_roc_curves.png
   02_confusion_matrices.png
   03_model_comparison_bar.png
   04_feature_importance_xgb_rf.png
   05_shap_beeswarm_xgb.png
   06_shap_bar_xgb.png
   07_shap_importance_xgb_rf_lr.png
   08_shap_waterfall_high_risk.png
   09_shap_waterfall_borderline.png
   10_shap_waterfall_confident_nonchurner.png
   11_shap_waterfall_missed_churner.png
   12_shap_dependence_top5.png
   13_shap_interaction_top2.png
   14_lime_local_high_risk.png
   15_lime_local_borderline.png
   16_lime_local_confident_nonchurner.png
   17_lime_local_missed_churner.png
   18_shap_vs_lime_importance.png
   19_fidelity_deletion_curves.png
   20_sparsity_histogram.png
   21_silhouette_elbow.png
   22_cluster_profiles_heatmap.png
   23_segments_pca_scatter.png

Expected 23. Download the figures/ folder -> code\outputs\figures\
